# 杭州租房列表爬虫（贝壳找房）

抓取贝壳找房杭州市租房列表页，提取标题、行政区、商圈、面积、月租金，并导出为 CSV。

> **说明**：贝壳对自动化访问有风控，若返回登录页，可在下方代码中填入浏览器 Cookie 后重试。

In [ ]:
import re
import time
import random

import pandas as pd
import requests
from lxml import etree

# ========== 可配置参数 ==========
BASE_URL = "https://hz.zu.ke.com/zufang"
MAX_PAGES = 5  # 抓取页数，可自行修改

# 若遇到登录/验证码页，从浏览器开发者工具复制 Cookie 粘贴到此处（字符串）
# 示例: COOKIE = "lianjia_uuid=xxx; lianjia_ssid=xxx; ..."
COOKIE = ""

# 常见浏览器 User-Agent 池（>=3 个）
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
]

In [ ]:
def build_page_url(page: int) -> str:
    """构造列表页 URL。第 1 页与第 n 页格式略有不同。"""
    if page == 1:
        return f"{BASE_URL}/"
    return f"{BASE_URL}/pg{page}/"


def build_headers() -> dict:
    """随机选取 User-Agent，并附带常见浏览器请求头。"""
    return {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
        "Connection": "keep-alive",
        "Referer": f"{BASE_URL}/",
        "Upgrade-Insecure-Requests": "1",
    }


def is_login_page(html: str) -> bool:
    """检测是否被重定向到登录/风控页。"""
    return 'ke-passport" content="LOGIN"' in html or "<title>登录</title>" in html


def parse_listing_item(item) -> dict | None:
    """用 XPath 从单个房源节点提取字段。"""
    title_nodes = item.xpath(".//p[contains(@class,'content__list--item--title')]//a/text()")
    if not title_nodes:
        return None

    title = title_nodes[0].strip()

    district_nodes = item.xpath(".//p[contains(@class,'content__list--item--des')]/a[1]/text()")
    biz_circle_nodes = item.xpath(".//p[contains(@class,'content__list--item--des')]/a[2]/text()")

    district = district_nodes[0].strip() if district_nodes else ""
    biz_circle = biz_circle_nodes[0].strip() if biz_circle_nodes else ""

    des_text = "".join(item.xpath(".//p[contains(@class,'content__list--item--des')]//text()"))
    area_match = re.search(r"([\d.]+)\s*㎡", des_text)
    area = area_match.group(1) if area_match else ""

    price_nodes = item.xpath(".//span[contains(@class,'content__list--item-price')]//em/text()")
    price = price_nodes[0].strip() if price_nodes else ""

    return {
        "title": title,
        "district": district,
        "biz_circle": biz_circle,
        "area": area,
        "price": price,
    }


def fetch_page(session: requests.Session, page: int) -> str:
    """请求单页 HTML。"""
    url = build_page_url(page)
    response = session.get(url, headers=build_headers(), timeout=20)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or "utf-8"
    return response.text


def parse_page(html: str) -> list[dict]:
    """解析一页中的所有房源。"""
    tree = etree.HTML(html)
    items = tree.xpath("//div[@class='content__list--item']")

    records = []
    for item in items:
        record = parse_listing_item(item)
        if record:
            records.append(record)
    return records

In [ ]:
session = requests.Session()
if COOKIE:
    session.headers.update({"Cookie": COOKIE})

all_records: list[dict] = []

for page in range(1, MAX_PAGES + 1):
    print(f"正在抓取第 {page}/{MAX_PAGES} 页 ...")

    html = fetch_page(session, page)

    if is_login_page(html):
        raise RuntimeError(
            "当前请求被重定向到登录页，请先在浏览器打开贝壳杭州租房页，"
            "复制 Cookie 填入上方 COOKIE 变量后重新运行。"
        )

    page_records = parse_page(html)
    print(f"  -> 本页解析到 {len(page_records)} 条房源")
    all_records.extend(page_records)

    if page < MAX_PAGES:
        sleep_seconds = random.uniform(1.5, 3.5)
        print(f"  -> 休眠 {sleep_seconds:.2f} 秒")
        time.sleep(sleep_seconds)

print(f"\n共抓取 {len(all_records)} 条记录")

In [ ]:
df = pd.DataFrame(all_records, columns=["title", "district", "biz_circle", "area", "price"])

output_file = "hangzhou_rent_raw.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"数据已保存至: {output_file}")
print("\n前 5 行预览:")
print(df.head())

# 数据预处理

读取 `hangzhou_rent_raw.csv`，完成缺失值/重复值处理、类型转换、衍生指标计算及 IQR 异常值剔除，输出 `hangzhou_rent_clean.csv`。

In [ ]:
import re

INPUT_FILE = "hangzhou_rent_raw.csv"
OUTPUT_FILE = "hangzhou_rent_clean.csv"


def strip_numeric(value) -> float:
    """剥离中文字符及单位，提取数值并转为 float。"""
    if pd.isna(value):
        return float("nan")
    text = str(value).strip()
    # 仅保留数字与小数点
    numeric_text = re.sub(r"[^\d.]", "", text)
    if not numeric_text:
        return float("nan")
    return float(numeric_text)


def remove_iqr_outliers(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """使用四分位距法（IQR）剔除指定列的异常值。"""
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]


# 1. 读取原始数据
df_raw = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
rows_before = len(df_raw)
print(f"清洗前行数: {rows_before}")
print(df_raw.head())

In [ ]:
# 2. 去除缺失值与完全重复行
df_clean = df_raw.dropna().drop_duplicates().copy()
print(f"去缺失/重复后行数: {len(df_clean)} (剔除 {rows_before - len(df_clean)} 行)")

# 3. 数据类型转换：剥离 area、price 中的中文及单位，转为 float
df_clean["area"] = df_clean["area"].apply(strip_numeric)
df_clean["price"] = df_clean["price"].apply(strip_numeric)

# 转换后若产生 NaN，再次剔除
df_clean = df_clean.dropna(subset=["area", "price"])
df_clean = df_clean[(df_clean["area"] > 0) & (df_clean["price"] > 0)]
print(f"类型转换后行数: {len(df_clean)}")

# 4. 计算衍生指标：每平米单价
df_clean["unit_price"] = (df_clean["price"] / df_clean["area"]).round(2)

In [ ]:
# 5. IQR 异常值处理（分别对 area、price 列检测并剔除）
rows_before_iqr = len(df_clean)

df_clean = remove_iqr_outliers(df_clean, "area")
print(f"area IQR 过滤后行数: {len(df_clean)} (剔除 {rows_before_iqr - len(df_clean)} 行)")

rows_before_price_iqr = len(df_clean)
df_clean = remove_iqr_outliers(df_clean, "price")
print(f"price IQR 过滤后行数: {len(df_clean)} (剔除 {rows_before_price_iqr - len(df_clean)} 行)")

rows_after = len(df_clean)

# 6. 输出对比与保存
print("\n========== 清洗前后行数对比 ==========")
print(f"清洗前: {rows_before} 行")
print(f"清洗后: {rows_after} 行")
print(f"共剔除: {rows_before - rows_after} 行")

df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\n干净数据已保存至: {OUTPUT_FILE}")
print("\n清洗后前 5 行预览:")
print(df_clean.head())

# 统计分析与可视化

基于 `hangzhou_rent_clean.csv`，对各行政区租金水平进行分组统计，并绘制箱线图与回归散点图。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 全局中文字体与负号显示设置
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

sns.set_theme(style="whitegrid", font="SimHei")

CLEAN_FILE = "hangzhou_rent_clean.csv"
df_analysis = pd.read_csv(CLEAN_FILE, encoding="utf-8-sig")

print(f"读取数据: {len(df_analysis)} 行")
df_analysis.head()

In [ ]:
# 各行政区平均租金、平均面积、平均单价
district_stats = (
    df_analysis.groupby("district", as_index=False)
    .agg(
        平均租金=("price", "mean"),
        平均面积=("area", "mean"),
        平均单价=("unit_price", "mean"),
    )
    .round(2)
    .sort_values("平均租金", ascending=False)
)

print("杭州各行政区租房指标统计:")
print(district_stats.to_string(index=False))

In [ ]:
# 图表 1：各行政区租金分布箱线图
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_analysis, x="district", y="price", palette="Set2")
plt.title("杭州各行政区租金分布对比", fontsize=14)
plt.xlabel("行政区")
plt.ylabel("租金（元/月）")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 图表 2：面积与租金回归散点图
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_analysis,
    x="area",
    y="price",
    scatter_kws={"alpha": 0.6, "s": 40},
    line_kws={"color": "red", "linewidth": 2},
)
plt.title("杭州租房面积与租金相关性分析", fontsize=14)
plt.xlabel("面积（㎡）")
plt.ylabel("租金（元/月）")
plt.tight_layout()
plt.show()

# KMeans 聚类分析

基于 `hangzhou_rent_clean.csv`，对面积、总租金、单价进行聚类，挖掘不同性价比房源特征。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

FEATURE_COLS = ["area", "price", "unit_price"]
K_RANGE = range(2, 7)  # K 从 2 到 6

# 读取清洗后的数据
df_cluster = pd.read_csv("hangzhou_rent_clean.csv", encoding="utf-8-sig")
X = df_cluster[FEATURE_COLS].copy()

# 数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"聚类特征: {FEATURE_COLS}")
print(f"样本数量: {len(df_cluster)}")

In [ ]:
# 遍历 K 值，计算轮廓系数并选择最优 K
silhouette_results = []

for k in K_RANGE:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_results.append({"K": k, "silhouette_score": round(score, 4)})
    print(f"K = {k}, 轮廓系数 = {score:.4f}")

silhouette_df = pd.DataFrame(silhouette_results)
best_k = int(silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "K"])
print(f"\n最优 K 值: {best_k}")

In [ ]:
# 使用最优 K 值执行最终聚类
final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster["cluster"] = final_kmeans.fit_predict(X_scaled)

# 各类群在面积、租金、单价上的均值
cluster_profile = (
    df_cluster.groupby("cluster")[FEATURE_COLS]
    .mean()
    .round(2)
    .rename(columns={
        "area": "平均面积",
        "price": "平均租金",
        "unit_price": "平均单价",
    })
)

print("各聚类簇特征均值:")
print(cluster_profile)
print("\n带聚类标签的数据预览:")
print(df_cluster.head())

In [ ]:
# 三维散点图：按聚类簇着色
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

colors = sns.color_palette("Set1", n_colors=best_k)
cluster_ids = sorted(df_cluster["cluster"].unique())

for cluster_id, color in zip(cluster_ids, colors):
    subset = df_cluster[df_cluster["cluster"] == cluster_id]
    ax.scatter(
        subset["area"],
        subset["price"],
        subset["unit_price"],
        c=[color],
        label=f"簇 {cluster_id}",
        s=50,
        alpha=0.8,
    )

ax.set_title("杭州租房 KMeans 聚类三维分布", fontsize=14)
ax.set_xlabel("面积（㎡）")
ax.set_ylabel("租金（元/月）")
ax.set_zlabel("单价（元/㎡）")
ax.legend(title="聚类簇")
plt.tight_layout()
plt.show()

# 空间可视化：杭州市租金单价热力地图

基于 `hangzhou_rent_clean.csv`，按行政区聚合平均租金单价，使用 pyecharts 绘制杭州市域热力地图。

In [ ]:
from pyecharts import options as opts
from pyecharts.charts import Map
from pyecharts.globals import CurrentConfig

# 使用在线地图资源（若本地未安装城市地图包，可自动加载）
CurrentConfig.ONLINE = True

MAP_OUTPUT_FILE = "hangzhou_rent_map.html"


def normalize_district(name: str) -> str:
    """规范行政区名称，确保带有「区/县/市」后缀以匹配 pyecharts 地图。"""
    name = str(name).strip()
    if name.endswith(("区", "县", "市")):
        return name
    return f"{name}区"


# 读取数据并按行政区计算平均租金单价
df_map = pd.read_csv("hangzhou_rent_clean.csv", encoding="utf-8-sig")
df_map["district_norm"] = df_map["district"].apply(normalize_district)

district_unit_price = (
    df_map.groupby("district_norm", as_index=False)["unit_price"]
    .mean()
    .round(2)
    .rename(columns={"district_norm": "district", "unit_price": "avg_unit_price"})
)

print("各行政区平均租金单价（元/㎡）:")
print(district_unit_price.to_string(index=False))

map_data = list(
    zip(
        district_unit_price["district"].tolist(),
        district_unit_price["avg_unit_price"].tolist(),
    )
)
min_price = float(district_unit_price["avg_unit_price"].min())
max_price = float(district_unit_price["avg_unit_price"].max())

In [ ]:
# 绘制杭州市域热力地图并保存为 HTML
hangzhou_map = (
    Map(init_opts=opts.InitOpts(width="1200px", height="800px", page_title="杭州租房单价地图"))
    .add(
        series_name="平均租金单价",
        data_pair=map_data,
        maptype="杭州",
        is_map_symbol_show=False,
        label_opts=opts.LabelOpts(is_show=True),
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="杭州市各行政区租金单价热力图",
            subtitle="颜色越深，平均租金单价越高（元/㎡）",
            pos_left="center",
        ),
        visualmap_opts=opts.VisualMapOpts(
            min_=min_price,
            max_=max_price,
            is_calculable=True,
            range_text=["高", "低"],
            pos_left="left",
            pos_bottom="10%",
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="item",
            formatter="{b}<br/>平均单价: {c} 元/㎡",
        ),
    )
)

hangzhou_map.render(MAP_OUTPUT_FILE)
print(f"地图已保存至: {MAP_OUTPUT_FILE}")
hangzhou_map